# Зависимость декодирования от вероятности ошибки

Здесь меняется вероятность ошибки `p`, а длина кода и лимит итераций остаются постоянными. Для каждого значения генерируется набор независимых кадров, все кадры обрабатываются Rust `decode-batch`. Мы измеряем долю точного восстановления, долю кадров с нулевым синдромом и среднее число итераций.

Демонстрация использует RA-6000. Увеличьте `FRAMES` для более надёжной оценки; для быстрых циклов отладки уменьшите его.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "Cargo.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Cargo.toml").exists():
    raise RuntimeError("Open this notebook from inside the ra-rust project")
sys.path.insert(0, str(ROOT / "python"))
from ra_reconcile import decode, decode_batch, encode
MATRIX_SMALL = ROOT / "data" / "H_small.edges"
MATRIX_RA = ROOT / "data" / "RA_n6000_R150.edges"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

SEED = 20261113
FRAMES = 16
P_VALUES = [0.002, 0.005, 0.01, 0.02]
THREADS = 4
MAX_ITER = 100
rng = np.random.default_rng(SEED)

# Fix one valid codeword; choose a random codeword once, then transmit it in all frames.
k = 120
information = "".join(rng.integers(0, 2, size=k).astype(str))
codeword = encode(MATRIX_RA, information)
print(f"n={len(codeword)}, k={k}, rate={k/len(codeword):.4f}; codeword cached for the sweep")

## Генерация канала и метрики

Пусть `E` — независимая маска ошибок с `P(Eⱼ=1)=p`, а `B` — случайный битовый поток Боба. Строим `A=B XOR E` и публичное сообщение `M=C XOR B`. Тогда `A XOR M=C XOR E`, что точно соответствует модели двоичного симметричного канала. Для каждого вывода считаем **точное восстановление** (`estimate == C`), а также `success` (прошёл синдром). Эти события не равнозначны.


In [ ]:
rows = []
for p in P_VALUES:
    errors = rng.random((FRAMES, len(codeword))) < p
    bobs = rng.integers(0, 2, size=errors.shape, dtype=np.uint8)
    c = np.fromiter((int(bit) for bit in codeword), dtype=np.uint8)
    alice = bobs ^ errors.astype(np.uint8)
    public = bobs ^ c
    a_frames = ["".join(row.astype(str)) for row in alice]
    m_frames = ["".join(row.astype(str)) for row in public]
    results = decode_batch(MATRIX_RA, a_frames, m_frames, p,
                           max_iter=MAX_ITER, threads=THREADS)
    exact = [r.estimate == codeword for r in results]
    rows.append({"p": p,
                 "exact_frame_rate": float(np.mean(exact)),
                 "syndrome_rate": float(np.mean([r.success for r in results])),
                 "mean_iterations": float(np.mean([r.iterations for r in results])),
                 "mean_ms_per_frame": float(np.mean([r.elapsed_ms for r in results]))})
    print(f"p={p:.3f}: exact={sum(exact)}/{FRAMES}, syndrome={sum(r.success for r in results)}/{FRAMES}")

summary = pd.DataFrame(rows)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(summary.p, summary.exact_frame_rate, "o-", label="точное слово")
axes[0].plot(summary.p, summary.syndrome_rate, "s--", label="нулевой синдром")
axes[0].set(xlabel="Вероятность ошибки p", ylabel="Доля кадров", ylim=(-0.05, 1.05))
axes[0].grid(alpha=.3); axes[0].legend()
axes[1].plot(summary.p, summary.mean_iterations, "o-", color="darkorange")
axes[1].set(xlabel="Вероятность ошибки p", ylabel="Среднее число итераций")
axes[1].grid(alpha=.3)
fig.suptitle("RA-6000: зависимость от качества канала")
fig.tight_layout()
plt.show()

## Интерпретация

Обычно с ростом `p` декодирование усложняется: возрастает число итераций и вероятность отказа. Конкретный порог зависит от матрицы и случайной выборки. При малом числе кадров точки шумные; повторите расчёт с большим `FRAMES` и новым `SEED`, чтобы оценить доверительный интервал.
